# 🧪 DPAC — Bayesian Exposure Assessment

**DPAC** (*Decision Point for Adequate Control*) = 70th posterior percentile of the exposure distribution.

**How to use:**
1. Run the cell below (Shift+Enter) — a configuration form will appear.
2. Choose the output folder, select your data files, and set the risk parameters.
3. Click **▶ Run Analysis** — the model runs for each file, the logfile is updated, and both graphs are generated automatically.

> Each data file must contain at least two columns: **`Valeur`** (measured concentration) and **`Capteur`** (sampler / group ID).
>
> Detection limits are read from **`LoD effectif`** and **`LoQ effectif`** columns **per row** (variable limits supported). If these columns are absent, bounds are parsed from censored value strings (`<val` → LoD, `[low-high]` → LoD/LoQ), falling back to defaults LoD = 0.01 and LoQ = 0.033.

In [ ]:
%matplotlib inline
import os, re
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import ipywidgets as w
from IPython.display import display

try:
    from ipyfilechooser import FileChooser
    HAS_FC = True
except ImportError:
    HAS_FC = False

# ── Core helpers ──────────────────────────────────────────────────────────────
def _find_col(df, *names):
    for name in names:
        for c in df.columns:
            if str(c).strip().lower() == name.strip().lower():
                return c
    return None

def _normal_cdf(x, mu, sigma):
    return 0.5 * (1 + pm.math.erf((x - mu) / (sigma * pm.math.sqrt(2))))

# ── openpyxl style constants ──────────────────────────────────────────────────
_XB  = Border(left=Side(style="thin", color="AAAAAA"), right=Side(style="thin", color="AAAAAA"),
              top=Side(style="thin",  color="AAAAAA"), bottom=Side(style="thin", color="AAAAAA"))
_XC  = Alignment(horizontal="center", vertical="center")
_XL  = Alignment(horizontal="left",   vertical="center")
_CW  = [28, 20, 6, 11, 10, 7, 9, 8, 9, 9, 10, 14, 14, 14, 14, 26, 28]
_DC  = set(range(11, 16)); _CC = 16; _DC2 = 17; _LM = "— LEGEND —"
_HDRS = ["File", "Test", "n", "Uncensored", "[LoQ-LoD]", "<LoD",
         "GM", "GSD", "GSDb", "GSDw", "DPAC",
         "DPAC_IC-Inf90", "DPAC_IC-Sup90", "DPAC_IC-Inf50", "DPAC_IC-Sup50",
         "MCMC Convergence", "Decision"]
_FOK = PatternFill("solid", fgColor="E2EFDA"); _FCA = PatternFill("solid", fgColor="FFF2CC")
_FPO = PatternFill("solid", fgColor="FFD7D7"); _FDP = PatternFill("solid", fgColor="FCE4D6")
_FHD = PatternFill("solid", fgColor="1F4E79"); _FQA = PatternFill("solid", fgColor="375623")
_FLG = PatternFill("solid", fgColor="D9D9D9"); _FOD = PatternFill("solid", fgColor="DEEAF1")
_FEV = PatternFill("solid", fgColor="FFFFFF")
_FH  = Font(name="Calibri", bold=True, color="FFFFFF", size=10)
_FD  = Font(name="Calibri", size=10)
_FDP_F = Font(name="Calibri", size=10, bold=True)
_FL  = Font(name="Calibri", size=9, italic=True)
_PAL = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2","#7f7f7f","#bcbd22","#17becf"]

# ── Logfile writer ────────────────────────────────────────────────────────────
def _write_log_row(logfile, nr_row):
    wb = load_workbook(logfile) if os.path.exists(logfile) else Workbook()
    ws = wb["DPAC Log"] if "DPAC Log" in wb.sheetnames else wb.create_sheet("DPAC Log")
    la = None
    for r in range(ws.max_row, 0, -1):
        if ws.cell(r, 1).value == _LM: la = r; break
    if la:
        df2 = la - 1 if la > 1 and all(ws.cell(la-1, c).value is None for c in range(1, 5)) else la
        ws.delete_rows(df2, ws.max_row - df2 + 1)
    emp = ws.cell(1, 1).value is None and ws.cell(2, 1).value is None
    nh  = ws.cell(2, 1).value == "File"
    sc  = max(ws.max_row + 5, 5); al = 0
    for r in range(sc, 0, -1):
        if any(ws.cell(r, c).value is not None for c in range(1, 3)): al = r; break
    if emp:
        ws.merge_cells(start_row=1, start_column=16, end_row=1, end_column=17)
        c2 = ws.cell(1, 16, "Quality Assessment")
        c2.font = Font(name="Calibri", bold=True, color="FFFFFF", size=10)
        c2.fill = _FQA; c2.alignment = _XC; c2.border = _XB
        ws.cell(1, 17).border = _XB; ws.row_dimensions[1].height = 16
        for ci, h in enumerate(_HDRS, 1):
            c2 = ws.cell(2, ci, h); c2.font = _FH; c2.fill = _FHD
            c2.alignment = _XC; c2.border = _XB
        ws.row_dimensions[2].height = 22
        for ci, cw in enumerate(_CW, 1):
            ws.column_dimensions[ws.cell(2, ci).column_letter].width = cw
        nxt = 3
    elif nh: nxt = max(3, al + 1)
    else:    nxt = max(2, al + 1)
    fr = _FOD if nxt % 2 == 0 else _FEV
    for ci, val in enumerate(nr_row, 1):
        c2 = ws.cell(nxt, ci, val); c2.border = _XB
        c2.alignment = _XL if ci <= 2 else _XC
        if ci in _DC:
            c2.fill = _FDP; c2.font = _FDP_F
        elif ci == _CC:
            c2.font = _FD; c2.fill = _FEV
        elif ci == _DC2:
            c2.font = Font(name="Calibri", size=10, bold=True)
            c2.fill = (_FOK if val == "Valide" else _FPO)
        else:
            c2.fill = fr; c2.font = _FD
    # Legend
    lr = ws.max_row + 2
    c2 = ws.cell(lr, 1, _LM); c2.font = Font(name="Calibri", bold=True, size=9)
    c2.fill = _FLG; c2.alignment = _XL; c2.border = _XB
    for ci2, lb in [(_CC, "MCMC Convergence"), (_DC2, "Decision")]:
        c2 = ws.cell(lr, ci2, lb); c2.font = Font(name="Calibri", bold=True, color="FFFFFF", size=9)
        c2.fill = _FHD; c2.alignment = _XC; c2.border = _XB
    lr += 1
    for fll, tc, td in [
        (_FOK, "R̂, divergences, ESS bulk / ESS tail",
               "Valide  —  R̂ ≤ 1.01, 0 div., ESS ≥ 400"),
        (_FPO, "",
               "Not applicable  —  R̂ > 1.01, ≥ 1 div., ou ESS < 400")]:
        for ci2, tx in [(_CC, tc), (_DC2, td)]:
            c2 = ws.cell(lr, ci2, tx); c2.font = _FL; c2.fill = fll
            c2.alignment = _XL; c2.border = _XB
        lr += 1
    wb.save(logfile)

def _write_risk_sheet(logfile, base, ps, mus, sts, mm, sm):
    wb  = load_workbook(logfile)
    rsn = base[:28] + "_risk"
    if rsn in wb.sheetnames: del wb[rsn]
    wsr = wb.create_sheet(rsn)
    for ci, h in enumerate(["OEL [ppm]","Overexp.(%)","CI_inf90","CI_sup90","CI_inf50","CI_sup50"], 1):
        c2 = wsr.cell(1, ci, h); c2.font = _FH; c2.fill = _FHD; c2.alignment = _XC; c2.border = _XB
    wsr.row_dimensions[1].height = 22
    for ci, cw in enumerate([12]*6, 1):
        wsr.column_dimensions[wsr.cell(1, ci).column_letter].width = cw
    oelg = np.logspace(np.log10(0.001), np.log10(np.percentile(ps, 99.999) * 2), 100)
    for r2, oel in enumerate(oelg, 2):
        fra2 = 1 - stats.norm.cdf((np.log(oel) - mus) / sts)
        pf   = lambda p, _f=fra2: float(np.clip(np.percentile(_f, p), 1e-9, 1-1e-9))
        row  = [round(float(oel), 4), round(float(np.mean(ps >= oel) * 100), 2),
                round(np.exp(mm + stats.norm.ppf(1-pf(95)) * sm), 4),
                round(np.exp(mm + stats.norm.ppf(1-pf( 5)) * sm), 4),
                round(np.exp(mm + stats.norm.ppf(1-pf(75)) * sm), 4),
                round(np.exp(mm + stats.norm.ppf(1-pf(25)) * sm), 4)]
        fr2  = _FOD if r2 % 2 == 0 else _FEV
        for ci, val in enumerate(row, 1):
            c2 = wsr.cell(r2, ci, val); c2.font = _FD; c2.fill = fr2; c2.alignment = _XC; c2.border = _XB
    wb.save(logfile)
    return rsn

# ── Widgets ───────────────────────────────────────────────────────────────────
_sl = {"description_width": "160px"}
_ll = w.Layout(width="480px")
_out = w.Output()

if HAS_FC:
    _w_dir = FileChooser(os.getcwd(), title="Output folder (logfile + graphs)", show_only_dirs=True)
else:
    _w_dir = w.Text(value=os.getcwd(), description="Output folder:", style=_sl, layout=_ll)

def _get_dir():
    if HAS_FC: return _w_dir.selected_path or os.getcwd()
    return _w_dir.value.strip() or os.getcwd()

_w_n = w.BoundedIntText(value=1, min=1, max=30, step=1,
                         description="Number of files:", style=_sl, layout=w.Layout(width="280px"))
_w_risk = w.BoundedFloatText(value=30.0, min=1, max=99, step=1,
                               description="Acceptable risk (%):", style=_sl, layout=w.Layout(width="280px"))
_w_pct  = w.BoundedFloatText(value=95.0, min=50, max=99.9, step=0.5,
                               description="Criterion percentile:", style=_sl, layout=w.Layout(width="280px"))

_file_pickers = []
_file_box = w.VBox([])

def _rebuild_pickers(change=None):
    global _file_pickers
    n = _w_n.value
    _file_pickers = []
    for i in range(n):
        if HAS_FC:
            fc = FileChooser(os.getcwd(), filter_pattern="*.xlsx", title=f"File {i+1}")
        else:
            fc = w.Text(placeholder="path/to/file.xlsx", description=f"File {i+1}:",
                        style={"description_width": "70px"}, layout=_ll)
        _file_pickers.append(fc)
    _file_box.children = tuple(_file_pickers)

_w_n.observe(_rebuild_pickers, names="value")
_rebuild_pickers()

_b_run = w.Button(description="▶  Run Analysis", button_style="success",
                   icon="play", layout=w.Layout(width="200px", height="42px"))

# ── Main callback ──────────────────────────────────────────────────────────────
_IV_RE = re.compile(r'^\[([0-9.]+)-([0-9.]+)\]$')

def _load_lod_loq(df, cd, cq, vs, nt, orig_idx_left, orig_idx_int):
    if cd:
        lod_arr = df[cd].astype(float).values.copy()
    else:
        lod_arr = np.full(nt, 0.01)
        for i in (np.concatenate([orig_idx_left, orig_idx_int])
                  if (len(orig_idx_left) + len(orig_idx_int)) > 0
                  else np.array([], dtype=int)):
            v = vs.iloc[i]
            if v.startswith("<"):
                try: lod_arr[i] = float(v[1:])
                except ValueError: pass
            else:
                m = _IV_RE.match(v)
                if m:
                    try: lod_arr[i] = float(m.group(1))
                    except ValueError: pass
    if cq:
        loq_arr = df[cq].astype(float).values.copy()
    else:
        loq_arr = np.full(nt, 0.033)
        for i in orig_idx_int:
            m = _IV_RE.match(vs.iloc[i])
            if m:
                try: loq_arr[i] = float(m.group(2))
                except ValueError: pass
    return lod_arr, loq_arr

def _on_run(b):
    out_dir = _get_dir()
    os.makedirs(out_dir, exist_ok=True)
    logfile  = os.path.join(out_dir, "DPAC_logfile.xlsx")
    risk_pct = _w_risk.value
    pct_crit = _w_pct.value
    z_crit   = stats.norm.ppf(pct_crit / 100)

    files = []
    for fp in _file_pickers:
        p = (fp.selected or "") if HAS_FC else fp.value.strip()
        if p and os.path.isfile(p): files.append(p)

    with _out:
        _out.clear_output()
        if not files:
            print("⚠️  No valid files selected."); return

        store = []

        for fi, fpath in enumerate(files):
            nom  = os.path.basename(fpath)
            base = os.path.splitext(nom)[0]
            print(f"\n{'='*60}\nFile {fi+1}/{len(files)}: {nom}\n{'='*60}")

            # ── Load data ────────────────────────────────────────────────────
            df  = pd.read_excel(fpath)
            cv  = _find_col(df, "Valeur"); cc = _find_col(df, "Capteur")
            cd  = _find_col(df, "LoD effectif", "LOD", "lod")
            cq  = _find_col(df, "LoQ effectif", "LOQ", "loq")
            if not cv or not cc:
                print("⚠️  Columns Valeur/Capteur not found — skipping."); continue

            gr   = df[cc].astype("category")
            gi, ng = gr.cat.codes.values, len(gr.cat.categories)
            vs   = df[cv].astype(str).str.strip()
            ml   = vs.str.startswith("<"); mi = vs.str.startswith("["); md = ~(ml | mi)
            orig_idx_int  = np.where(mi)[0]
            orig_idx_left = np.where(ml)[0]
            ly   = np.log(df.loc[md, cv].astype(float).values)
            ni, nl, nt = int(mi.sum()), int(ml.sum()), len(df)

            lod_arr, loq_arr = _load_lod_loq(df, cd, cq, vs, nt, orig_idx_left, orig_idx_int)
            log_LOD = np.log(lod_arr); log_LOQ = np.log(loq_arr)

            n_lod_u = len(np.unique(lod_arr)); n_loq_u = len(np.unique(loq_arr))
            lod_info = (f"col '{cd}' ({'unique: ' + str(lod_arr[0]) if n_lod_u == 1 else f'variable: {n_lod_u} values, {lod_arr.min():.4g}–{lod_arr.max():.4g}'})"
                        if cd else "default=0.01")
            loq_info = (f"col '{cq}' ({'unique: ' + str(loq_arr[0]) if n_loq_u == 1 else f'variable: {n_loq_u} values, {loq_arr.min():.4g}–{loq_arr.max():.4g}'})"
                        if cq else "default=0.033")
            print(f"n={nt}  detected={len(ly)}  [LoD-LoQ]={ni}  <LoD={nl}")
            print(f"LoD: {lod_info}"); print(f"LoQ: {loq_info}")
            print("MCMC sampling (5–15 min) …")

            # ── PyMC model ───────────────────────────────────────────────────
            with pm.Model():
                mu  = pm.Uniform("mu", lower=-20, upper=20)
                sB  = pm.LogNormal("sigma_B", mu=-0.8786, sigma=0.7823012)
                sW  = pm.LogNormal("sigma_W", mu=-0.4106, sigma=0.7254381)
                zi  = pm.Normal("z_i", mu=0, sigma=1, shape=ng)
                mui = pm.Deterministic("mu_i", mu + zi * sB)
                pm.Normal("obs", mu=mui[gi[md]], sigma=sW, observed=ly)
                if ni > 0:
                    for k, cix in enumerate(gi[mi]):
                        p = (_normal_cdf(log_LOQ[orig_idx_int[k]], mui[cix], sW) -
                             _normal_cdf(log_LOD[orig_idx_int[k]], mui[cix], sW))
                        pm.Potential(f"ci_{k}", pm.math.log(p + 1e-10))
                if nl > 0:
                    for k, cix in enumerate(gi[ml]):
                        pm.Potential(f"cl_{k}", pm.math.log(
                            _normal_cdf(log_LOD[orig_idx_left[k]], mui[cix], sW) + 1e-10))
                st = pm.Deterministic("sigma_total", pm.math.sqrt(sB**2 + sW**2))
                _  = pm.Deterministic("p_critere",   pm.math.exp(mu + z_crit * st))
                trc = pm.sample(4000, tune=2000, chains=4, target_accept=0.95,
                                random_seed=42, progressbar=True, nuts_sampler="numpyro")

            # ── MCMC diagnostics ─────────────────────────────────────────────
            ps  = trc.posterior["p_critere"].values.flatten()
            mus = trc.posterior["mu"].values.flatten()
            sBs = trc.posterior["sigma_B"].values.flatten()
            sWs = trc.posterior["sigma_W"].values.flatten()
            sts = trc.posterior["sigma_total"].values.flatten()
            ndiv = int(trc.sample_stats["diverging"].values.sum())
            rh   = az.rhat(trc, var_names=["mu","sigma_B","sigma_W","p_critere"])
            mrh  = float(np.max([float(v.values) for v in rh.data_vars.values()]))
            ess_b    = az.ess(trc, var_names=["p_critere"])
            ess_t    = az.ess(trc, var_names=["p_critere"], method="tail")
            ess_bulk = int(float(np.asarray(ess_b["p_critere"].values).ravel()[0]))
            ess_tail = int(float(np.asarray(ess_t["p_critere"].values).ravel()[0]))
            min_ess  = min(ess_bulk, ess_tail)
            conv = f"R̂={mrh:.3f}, {ndiv} div., ESS={ess_bulk}/{ess_tail}"
            dec  = "Valide" if ndiv == 0 and mrh <= 1.01 and min_ess >= 400 else "Not applicable"

            # ── Compute DPAC and CI ──────────────────────────────────────────
            dpac = np.percentile(ps, 100 - risk_pct)
            mm, sm = np.median(mus), np.median(sts)
            fra  = 1 - stats.norm.cdf((np.log(dpac) - mus) / sts)
            i90  = np.exp(mm + stats.norm.ppf(1 - np.percentile(fra, 95)) * sm)
            s90  = np.exp(mm + stats.norm.ppf(1 - np.percentile(fra,  5)) * sm)
            i50  = np.exp(mm + stats.norm.ppf(1 - np.percentile(fra, 75)) * sm)
            s50  = np.exp(mm + stats.norm.ppf(1 - np.percentile(fra, 25)) * sm)
            ess_ok = "OK" if min_ess >= 400 else "Attention : ESS < 400 — convergence insuffisante"
            print(f"\n✓ DPAC = {dpac:.4f} ppm  |  {conv}  |  {dec}")
            print(f"  ESS bulk : {ess_bulk}  |  ESS tail : {ess_tail}  →  {ess_ok}")

            nr_row = [base, "", nt, len(ly), ni, nl,
                      round(float(np.exp(mm)),4), round(float(np.exp(sm)),2),
                      round(float(np.exp(np.median(sBs))),4),
                      round(float(np.exp(np.median(sWs))),4),
                      round(float(dpac),4),
                      round(float(i90),4), round(float(s90),4),
                      round(float(i50),4), round(float(s50),4),
                      conv, dec]
            _write_log_row(logfile, nr_row)
            rsn = _write_risk_sheet(logfile, base, ps, mus, sts, mm, sm)
            print(f"✓ Logfile updated  →  {logfile}")
            store.append({"base": base, "dpac": dpac, "ps": ps, "mus": mus, "sts": sts, "mm": mm, "sm": sm})

        if not store:
            print("\n⚠️  No results to plot."); return

        print(f"\n{'='*60}\nGenerating graphs …")

        # ── Risk curves ────────────────────────────────────────────────────────
        xl_f   = pd.ExcelFile(logfile)
        rshs   = [s for s in xl_f.sheet_names if s.lower().endswith("_risk")]
        peek   = pd.read_excel(logfile, sheet_name="DPAC Log", header=None, nrows=2)
        hrow   = 1 if str(peek.iloc[1, 0]).strip() == "File" else 0
        df_log = pd.read_excel(logfile, sheet_name="DPAC Log", header=hrow)

        if rshs:
            fig3, ax3 = plt.subplots(figsize=(10, 6))
            lbls3 = []
            for idx, sn in enumerate(rshs):
                df_r  = pd.read_excel(logfile, sheet_name=sn)
                clr   = _PAL[idx % len(_PAL)]
                label = re.sub(r"_risk$", "", sn, flags=re.IGNORECASE)
                ax3.plot(df_r.iloc[:,0], df_r.iloc[:,1], color=clr, lw=2, label=label)
                ax3.fill_betweenx(df_r.iloc[:,1], df_r.iloc[:,2], df_r.iloc[:,3], color=clr, alpha=0.12)
                ax3.fill_betweenx(df_r.iloc[:,1], df_r.iloc[:,4], df_r.iloc[:,5], color=clr, alpha=0.25)
                _dpac_map = {s["base"][:28]: s["dpac"] for s in store}
                _base_lbl = label[:28]
                if _base_lbl in _dpac_map:
                    dv    = float(_dpac_map[_base_lbl])
                    ov_at = float(np.interp(dv, df_r.iloc[:,0].values, df_r.iloc[:,1].values))
                    ax3.annotate(f"DPAC={dv:.1f}", xy=(dv, ov_at),
                                 xytext=(dv*1.5, min(ov_at+12, 92)),
                                 arrowprops=dict(arrowstyle="->", color=clr),
                                 fontsize=8, color=clr,
                                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=clr, alpha=0.9))
                lbls3.append(label)
            rf = f"{risk_pct:.0f}" if risk_pct==int(risk_pct) else f"{risk_pct}"
            ax3.axhline(y=risk_pct, color="green", ls="--", lw=1, label=f"Risk = {rf}%")
            ax3.set_xscale("log"); ax3.set_ylim(-5, 105)
            ax3.set_xlabel("OEL [ppm]"); ax3.set_ylabel("Risk (%)")
            ax3.grid(True, which="both", ls="--", alpha=0.5)
            ax3.legend(loc="upper center", bbox_to_anchor=(0.5,-0.16),
                       ncol=min(3, len(lbls3)+1), fontsize=8, frameon=False)
            plt.tight_layout(); fig3.subplots_adjust(bottom=0.28); plt.show()
            out3 = os.path.join(out_dir, "DPAC_risk_curves")
            fig3.savefig(out3+".png", dpi=300, bbox_inches="tight")
            fig3.savefig(out3+".pdf", bbox_inches="tight")
            print(f"✓ Risk curves  →  {out3}.png / .pdf")

        # ── Comparative ────────────────────────────────────────────────────────
        df_c = df_log.rename(columns={df_log.columns[0]: "Label"})
        df_c = df_c.rename(columns={"DPAC_IC-Inf90":"Inf90","DPAC_IC-Sup90":"Sup90",
                                     "DPAC_IC-Inf50":"Inf50","DPAC_IC-Sup50":"Sup50"})
        for col in ["Inf90","Sup90","Inf50","Sup50"]:
            df_c[col] = pd.to_numeric(df_c[col], errors="coerce")
        df_c = df_c[(df_c["Sup90"]>df_c["Inf90"]) & (df_c["Sup50"]>df_c["Inf50"])].reset_index(drop=True)

        if len(df_c):
            ic90c, c50c = "#1f77b4", "#d62728"
            hd  = "DPAC" in df_c.columns
            n_c = len(df_c)
            fig4, ax4 = plt.subplots(figsize=(11, max(3.5, 0.75*n_c+1.5)))
            for i, row in df_c.iterrows():
                if row["Inf50"] > row["Inf90"]: ax4.hlines(i, row["Inf90"], row["Inf50"], lw=6, colors=ic90c)
                ax4.hlines(i, row["Inf50"], row["Sup50"], lw=6, colors=c50c)
                if row["Sup90"] > row["Sup50"]: ax4.hlines(i, row["Sup50"], row["Sup90"], lw=6, colors=ic90c)
                if hd and pd.notna(row.get("DPAC")):
                    dv = float(row["DPAC"])
                    ax4.plot(dv, i, "v", color="black", markersize=8, zorder=6)
                    ax4.text(dv, i+0.35, f"{dv:.1f}", ha="center", va="center",
                             fontsize=10, fontweight="bold", color="black")
            ax4.set_yticks(range(n_c)); ax4.set_yticklabels(df_c["Label"].tolist(), fontsize=10)
            ax4.invert_yaxis(); ax4.set_ylim(n_c-1+0.9, -0.9)
            rf = f"{risk_pct:.0f}" if risk_pct==int(risk_pct) else f"{risk_pct}"
            ax4.set_xscale("log")
            ax4.set_xlabel(f"DPAC (Risk {rf}%)  [ppm]", fontsize=10)
            ax4.set_ylabel("Dataset", fontsize=10)
            ax4.grid(axis="x", ls="--", alpha=0.35)
            xmn, xmx = df_c["Inf90"].min(), df_c["Sup90"].max()
            xp = (xmx/xmn)**0.06 if xmx>xmn else 1.15
            ax4.set_xlim(xmn/xp, xmx*xp)
            lh = [Line2D([0],[0], color=ic90c, lw=5, label="90% Credible Interval"),
                  Line2D([0],[0], color=c50c,  lw=5, label="50% Credible Interval")]
            if hd: lh.append(Line2D([0],[0], marker="v", color="black", lw=0, markersize=8, label="DPAC"))
            ax4.legend(handles=lh, loc="upper center", bbox_to_anchor=(0.5,-0.12),
                       ncol=2, fontsize=11, frameon=False)
            plt.tight_layout(); fig4.subplots_adjust(bottom=0.20); plt.show()
            out4 = os.path.join(out_dir, "DPAC_comparative")
            fig4.savefig(out4+".png", dpi=300, bbox_inches="tight")
            fig4.savefig(out4+".pdf", bbox_inches="tight")
            print(f"✓ Comparative  →  {out4}.png / .pdf")

        print(f"\n{'='*60}")
        print(f"✅  Done!   Logfile : {logfile}")

_b_run.on_click(_on_run)

# ── Display ───────────────────────────────────────────────────────────────────
display(w.VBox([
    w.HTML("<h3 style='color:#1F4E79;margin:4px 0'>⚙️  Configuration</h3>"),
    w.HTML("<b>Output folder</b> (logfile + graphs):"),
    _w_dir,
    w.HTML("<br>"),
    _w_n,
    w.HTML("<br><b>Data files to analyse:</b>"),
    _file_box,
    w.HTML("<br>"),
    w.HBox([_w_risk, _w_pct]),
    w.HTML("<br>"),
    _b_run,
    _out,
]))